In [2]:
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

NUM_COURSES, NUM_SECTIONS, NUM_PROFESSORS = 15, 10, 10
NUM_DAYS, NUM_TIMESLOTS, NUM_ROOMS = 5, 6, 30
DAY_PAIRS = [(0, 2), (0, 3), (0, 4), (1, 3), (1, 4), (2, 4)]

class Room:
    def __init__(self, room_id, size):
        self.id = room_id
        self.size = size

class ClassSection:
    def __init__(self, section_id, course_id, prof_id, size):
        self.id = section_id
        self.course_id = course_id
        self.prof_id = prof_id
        self.size = size

rooms = [Room(i, random.choice([0, 1])) for i in range(NUM_ROOMS)]
classes = []
prof_course_count = {i: 0 for i in range(1, NUM_PROFESSORS + 1)}

for section_id in range(25):
    course_id = random.randint(1, NUM_COURSES)
    available_profs = [p for p, count in prof_course_count.items() if count < 3]
    if not available_profs:
        break
    prof_id = random.choice(available_profs)
    prof_course_count[prof_id] += 1
    classes.append(ClassSection(section_id, course_id, prof_id, random.choice([0, 1])))



In [3]:
class Schedule:
    def __init__(self):
        self.genes = {}
        self.fitness = 0
        self.conflicts = 0

    def random_init(self):
        for c in classes:
            self.genes[c.id] = [
                random.randint(0, len(DAY_PAIRS) - 1),
                random.randint(0, NUM_TIMESLOTS - 1),
                random.randint(0, NUM_ROOMS - 1),
                random.randint(0, NUM_TIMESLOTS - 1),
                random.randint(0, NUM_ROOMS - 1)
            ]
        self.calculate_fitness()

    def calculate_fitness(self):
        self.conflicts = 0
        room_usage = set()
        prof_usage = set()

        for c in classes:
            dp_idx, slot1, room1_idx, slot2, room2_idx = self.genes[c.id]
            day1, day2 = DAY_PAIRS[dp_idx]
            
            if rooms[room1_idx].size < c.size: self.conflicts += 1
            if rooms[room2_idx].size < c.size: self.conflicts += 1
            
            for d, s, r in [(day1, slot1, room1_idx), (day2, slot2, room2_idx)]:
                if (d, s, r) in room_usage: self.conflicts += 1
                else: room_usage.add((d, s, r))
                
            for d, s in [(day1, slot1), (day2, slot2)]:
                if (d, s, c.prof_id) in prof_usage: self.conflicts += 1
                else: prof_usage.add((d, s, c.prof_id))
                
        self.fitness = 1.0 / (1.0 + self.conflicts)

def crossover(parent1, parent2):
    child = Schedule()
    for c in classes:
        child.genes[c.id] = parent1.genes[c.id].copy() if random.random() > 0.5 else parent2.genes[c.id].copy()
    return child

def mutate(schedule, mutation_rate=0.1):
    for c in classes:
        if random.random() < mutation_rate:
            idx = random.randint(0, 4)
            limits = [len(DAY_PAIRS) - 1, NUM_TIMESLOTS - 1, NUM_ROOMS - 1, NUM_TIMESLOTS - 1, NUM_ROOMS - 1]
            schedule.genes[c.id][idx] = random.randint(0, limits[idx])
    schedule.calculate_fitness()



In [4]:
POPULATION_SIZE = 100
GENERATIONS = 500

population = [Schedule() for _ in range(POPULATION_SIZE)]
for p in population: p.random_init()

best_fitness_history = []
best_conflicts_history = []

for gen in range(GENERATIONS):
    population.sort(key=lambda x: x.fitness, reverse=True)
    best_fitness_history.append(population[0].fitness)
    best_conflicts_history.append(population[0].conflicts)
    
    if population[0].conflicts == 0:
        break
        
    new_population = [population[0]]
    while len(new_population) < POPULATION_SIZE:
        p1 = max(random.sample(population, 3), key=lambda x: x.fitness)
        p2 = max(random.sample(population, 3), key=lambda x: x.fitness)
        
        child = crossover(p1, p2)
        mutate(child)
        child.calculate_fitness()
        new_population.append(child)
        
    population = new_population

best_schedule = population[0]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, len(best_conflicts_history))
ax.set_ylim(0, max(best_conflicts_history) + 2)
ax.set_xlabel("Thế hệ (Generation)")
ax.set_ylabel("Số lỗi (Conflicts)")
ax.grid(True, linestyle='--', alpha=0.6)

line, = ax.plot([], [], color='red', linewidth=2)

def update(frame):
    line.set_data(range(frame), best_conflicts_history[:frame])
    
    current_conflict = best_conflicts_history[frame-1] if frame > 0 else best_conflicts_history[0]
    ax.set_title(f"Quá trình tối ưu GA - Thế hệ {frame} | Số lỗi: {current_conflict}")
    return line,

print("Đang tạo file GIF, vui lòng đợi...")
ani = animation.FuncAnimation(fig, update, frames=len(best_conflicts_history), blit=True)

gif_filename = 'ga_scheduling_progress.gif'
ani.save(gif_filename, writer='pillow', fps=6) 

plt.close(fig)
print(f"[*] Đã xuất thành công file: {gif_filename}")

Đang tạo file GIF, vui lòng đợi...
[*] Đã xuất thành công file: ga_scheduling_progress.gif
